# AI Video Watermark Removal - Google Colab

This notebook runs the complete watermark removal pipeline in Google Colab.

**Features:**
- Install IOPaint and dependencies
- Upload video via UI
- Auto-detect and remove watermark
- Download cleaned video

## 1. Install Dependencies

In [ ]:
# Install IOPaint and required packages
!pip install iopaint opencv-python-headless numpy requests tqdm scikit-image -q

# Install FFmpeg
!apt-get update -qq && apt-get install -y -qq ffmpeg

print("Dependencies installed!")

## 2. Start IOPaint Server

Run IOPaint in the background using LaMA model (CPU-optimized).

In [ ]:
import subprocess
import time

# Start IOPaint server in background
process = subprocess.Popen(
    ["iopaint", "start", "--model", "lama", "--device", "cpu", "--port", "8080"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for server to start
time.sleep(10)
print("IOPaint server started on port 8080")

## 3. Upload Video

Upload your video file using the file picker below.

In [ ]:
from google.colab import files
import shutil
import os

# Upload video
uploaded = files.upload()

# Get uploaded filename
input_video = list(uploaded.keys())[0]
print(f"Uploaded: {input_video}")

## 4. Run Watermark Removal Pipeline

In [ ]:
import sys
import cv2
import numpy as np
import requests
import io
from pathlib import Path
from tqdm import tqdm

# Setup paths
frames_dir = "/content/frames"
clean_frames_dir = "/content/clean_frames"
output_video = "/content/output_cleaned.mp4"

os.makedirs(frames_dir, exist_ok=True)
os.makedirs(clean_frames_dir, exist_ok=True)

API_URL = "http://127.0.0.1:8080/api/v1/inpaint"

def extract_frames(video_path, output_dir, resize=(640, 360)):
    """Extract frames using FFmpeg."""
    import subprocess
    
    # Clean existing
    for f in Path(output_dir).glob("*.png"):
        f.unlink()
    
    cmd = [
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
        "-i", video_path,
        "-vf", f"scale={resize[0]}:{resize[1]}:flags=lanczos",
        "-pix_fmt", "rgb24",
        f"{output_dir}/frame_%04d.png"
    ]
    
    subprocess.run(cmd, check=True)
    frame_files = sorted(Path(output_dir).glob("frame_*.png"))
    print(f"Extracted {len(frame_files)} frames")
    return frame_files

def detect_watermark(frame_files, sample_frames=15):
    """Auto-detect watermark using low-variance analysis."""
    sample = min(sample_frames, len(frame_files))
    indices = np.linspace(0, len(frame_files)-1, sample, dtype=int)
    
    gray_frames = []
    for idx in indices:
        frame = cv2.imread(str(frame_files[idx]))
        if frame is not None:
            gray_frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
    
    if len(gray_frames) < 3:
        return None
    
    # Stack and compute variance
    stack = np.stack(gray_frames, axis=0).astype(np.float32)
    variance_map = np.var(stack, axis=0)
    
    # Threshold low variance
    var_norm = cv2.normalize(variance_map, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    _, binary = cv2.threshold(var_norm, 15, 255, cv2.THRESH_BINARY_INV)
    
    # Morphology
    kernel = np.ones((5, 5), np.uint8)
    opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    dilated = cv2.dilate(opened, np.ones((10, 10), np.uint8), iterations=3)
    
    # Find largest contour
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    
    largest = max(contours, key=cv2.contourArea)
    if cv2.contourArea(largest) < 500:
        return None
    
    return cv2.boundingRect(largest)

def inpaint_frame(frame, mask):
    """Send frame to IOPaint API."""
    _, img_encoded = cv2.imencode(".png", frame)
    _, mask_encoded = cv2.imencode(".png", mask)
    
    files = {
        "image": ("image.png", io.BytesIO(img_encoded.tobytes()), "image/png"),
        "mask": ("mask.png", io.BytesIO(mask_encoded.tobytes()), "image/png")
    }
    
    response = requests.post(API_URL, files=files, data={"model": "lama"}, timeout=120)
    
    if response.status_code != 200:
        return None
    
    result = np.frombuffer(response.content, np.uint8)
    return cv2.imdecode(result, cv2.IMREAD_COLOR)

def rebuild_video(frame_dir, output_path, fps=30):
    """Rebuild video from frames."""
    import subprocess
    
    cmd = [
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
        "-framerate", str(fps),
        "-i", f"{frame_dir}/frame_%04d.png",
        "-c:v", "libx264",
        "-crf", "18",
        "-preset", "medium",
        "-pix_fmt", "yuv420p",
        "-an",
        output_path
    ]
    
    subprocess.run(cmd, check=True)
    print(f"Video saved: {output_path}")

# Run pipeline
print("Step 1: Extracting frames...")
frame_files = extract_frames(input_video, frames_dir)

print("\nStep 2: Detecting watermark...")
bbox = detect_watermark(frame_files)

if bbox:
    print(f"Watermark detected at: {bbox}")
else:
    print("No watermark detected, processing anyway...")
    bbox = (0, 0, 0, 0)

print("\nStep 3: Processing frames...")
for i, frame_path in enumerate(tqdm(frame_files, desc="Inpainting")):
    frame = cv2.imread(str(frame_path))
    if frame is None:
        continue
    
    h, w = frame.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    
    if bbox != (0, 0, 0, 0):
        x, y, bw, bh = bbox
        cv2.rectangle(mask, (x, y), (x+bw, y+bh), 255, -1)
        
        clean_frame = inpaint_frame(frame, mask)
        if clean_frame is not None:
            frame = clean_frame
    
    out_path = Path(clean_frames_dir) / frame_path.name
    cv2.imwrite(str(out_path), frame)

print("\nStep 4: Rebuilding video...")
rebuild_video(clean_frames_dir, output_video)

print("\nDone! Download your video below.")

## 5. Download Result

In [ ]:
from google.colab import files

# Download cleaned video
files.download(output_video)
print("Download started!")

## Optional: Mount Google Drive

Instead of uploading, you can process videos from your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Example: Process video from Drive
# input_video = "/content/drive/MyDrive/videos/input.mp4"
# output_video = "/content/drive/MyDrive/videos/output_cleaned.mp4"